# GDL Active Learning Dashboard
Run this notebook after updating `data/raw/` and `src/config.py`.
The notebook reports which features and target are active, then fits the GP and proposes new experiments when the bounds are complete.

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams

rcParams.update({
    "figure.dpi": 120,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "legend.fontsize": 9,
    "figure.facecolor": "white",
})

def _find_project_root():
    start_dirs = []
    nb = globals().get("__vsc_ipynb_file__")
    if nb:
        start_dirs.append(os.path.dirname(os.path.realpath(nb)))
    start_dirs.append(os.getcwd())
    for start in start_dirs:
        d = os.path.abspath(start)
        for _ in range(10):
            if os.path.isdir(os.path.join(d, "src")) and os.path.isdir(os.path.join(d, "data")):
                return d
            d = os.path.dirname(d)
    raise RuntimeError("Cannot find pemfc-gp project root (expected src/ and data/)")

PROJECT_ROOT = _find_project_root()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")

from src.config import (
    FEATURE_COLUMNS,
    OPTIMIZATION_GOAL,
    TARGET_COLUMN,
    get_feature_bounds,
    get_feature_labels,
    get_target_label,
)
from src.data_parser import load_and_sanitize_data
from src.gp_model import FuelCellSurrogate
from src.optimizer import get_batch_experiments, get_next_experiment

X_train, y_train = load_and_sanitize_data("data/raw/", "data/processed/")
print(f"Loaded {len(y_train)} observations. Arrays saved to data/processed/.")


In [ ]:
feature_labels = get_feature_labels()
target_label = get_target_label()
feature_to_index = {name: idx for idx, name in enumerate(FEATURE_COLUMNS)}
constant_features = []

print("=== ACTIVE CONFIGURATION ===")
print("Active features:", FEATURE_COLUMNS)
print("Active target:", TARGET_COLUMN)
print("Optimization goal:", OPTIMIZATION_GOAL)
print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}\n")

for idx, (feature_name, feature_label) in enumerate(zip(FEATURE_COLUMNS, feature_labels)):
    values = X_train[:, idx]
    unique_count = np.unique(values).size
    is_constant = np.isclose(values.min(), values.max())
    if is_constant:
        constant_features.append(feature_name)
    print(
        f"- {feature_label}: min={values.min():.4f}, max={values.max():.4f}, "
        f"mean={values.mean():.4f}, unique={unique_count}, constant={is_constant}"
    )

print(f"\n{target_label}: min={y_train.min():.4f}, max={y_train.max():.4f}, mean={y_train.mean():.4f}")

try:
    bounds = get_feature_bounds()
    print("\nConfigured optimizer bounds:")
    for feature_name, feature_label, bound in zip(FEATURE_COLUMNS, feature_labels, bounds):
        print(f"- {feature_label}: {bound}")
except KeyError as exc:
    bounds = None
    print("\nOptimizer bounds are incomplete in src/config.py:")
    print(exc)

if constant_features:
    print("\nConstant features in the current training data:", constant_features)

if len(FEATURE_COLUMNS) != 2:
    print("\nVisualization note: contour plots are only available when exactly two active features are configured.")


In [ ]:
model = FuelCellSurrogate()
model.fit(X_train, y_train)
print(f"GP fitted with {model.n_features} active feature(s).")
print(f"Optimized kernel: {model.gp.kernel_}")
print(f"Log-marginal likelihood: {model.gp.log_marginal_likelihood_value_:.3f}\n")

print("Input scaler mean / scale by active feature:")
for feature_name, feature_label, mean_value, scale_value in zip(
    FEATURE_COLUMNS, feature_labels, model.scaler_X.mean_, model.scaler_X.scale_
):
    print(f"- {feature_label}: mean={mean_value:.4f}, scale={scale_value:.4f}")

y_true, y_pred_loo, y_std_loo, rmse = model.loo_cross_validate(X_train, y_train)
print(f"\nLOO RMSE: {rmse:.4f} ({100 * rmse / y_train.mean():.1f}% of mean target)")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

vmin = min(y_true.min(), y_pred_loo.min()) - 0.02
vmax = max(y_true.max(), y_pred_loo.max()) + 0.02
ax1.plot([vmin, vmax], [vmin, vmax], "k--", lw=1, label="Perfect")
ax1.errorbar(
    y_true,
    y_pred_loo,
    yerr=1.96 * y_std_loo,
    fmt="o",
    color="#2196F3",
    ecolor="#90CAF9",
    elinewidth=1.5,
    capsize=3,
    ms=6,
    mec="black",
    mew=0.5,
    label="LOO ± 95% CI",
)
ax1.set(
    xlabel=f"Actual {TARGET_COLUMN}",
    ylabel=f"Predicted {TARGET_COLUMN}",
    title=f"LOO CV (RMSE = {rmse:.4f})",
    xlim=(vmin, vmax),
    ylim=(vmin, vmax),
)
ax1.set_aspect("equal")
ax1.legend(loc="upper left")
ax1.grid(True, alpha=0.3)

residuals = y_pred_loo - y_true
colors = ["#EF5350" if residual < 0 else "#66BB6A" for residual in residuals]
ax2.bar(range(len(residuals)), residuals, color=colors, edgecolor="black", lw=0.4)
ax2.axhline(0, color="black", lw=0.8)
ax2.set(xlabel="Observation index", ylabel="Residual", title="LOO residuals")
ax2.grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
next_point = None
X_grid = None
ei = None
batch = np.empty((0, len(FEATURE_COLUMNS)))

if bounds is None:
    print("Optimizer skipped: add FEATURE_BOUNDS for every active feature in src/config.py.")
else:
    next_point, X_grid, ei = get_next_experiment(model, X_train, y_train, bounds)
    print(f"TOP RECOMMENDATION ({OPTIMIZATION_GOAL})")
    for feature_name, feature_label, value in zip(FEATURE_COLUMNS, feature_labels, next_point):
        print(f"- {feature_label}: {value:.4f}")

    batch, _, _ = get_batch_experiments(
        model,
        X_train,
        y_train,
        bounds,
        k=5,
        min_distance=0.15,
    )

    print("\nBATCH SUGGESTIONS")
    for row_index, point in enumerate(batch, start=1):
        print(f"{row_index}.")
        for feature_name, feature_label, value in zip(FEATURE_COLUMNS, feature_labels, point):
            print(f"   - {feature_label}: {value:.4f}")


In [ ]:
if next_point is None:
    print("Visualization skipped because the optimizer did not run.")
elif len(FEATURE_COLUMNS) != 2:
    print("2D contour visualization skipped because exactly two active features are required.")
    print("Active features:", FEATURE_COLUMNS)
else:
    res = 100
    x_label, y_label = feature_labels[:2]
    x_name, y_name = FEATURE_COLUMNS[:2]

    x_plot = np.linspace(*bounds[0], res)
    y_plot = np.linspace(*bounds[1], res)
    x_mesh, y_mesh = np.meshgrid(x_plot, y_plot)
    X_plot = np.c_[x_mesh.ravel(), y_mesh.ravel()]

    mu_plot, std_plot = model.predict(X_plot, return_std=True)
    mu_mesh = mu_plot.reshape(res, res)
    std_mesh = std_plot.reshape(res, res)
    ei_mesh = ei.reshape(res, res)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5.5), constrained_layout=True)
    sc_kw = dict(edgecolors="black", linewidths=0.6, zorder=5)

    ax = axes[0]
    cf = ax.contourf(x_mesh, y_mesh, mu_mesh, levels=40, cmap="viridis")
    fig.colorbar(cf, ax=ax, label=target_label, shrink=0.85)
    ax.scatter(X_train[:, 0], X_train[:, 1], c="white", s=45, marker="o", label="Observations", **sc_kw)
    ax.scatter(*next_point, c="cyan", s=220, marker="*", label="Best next", **sc_kw)
    if len(batch) > 1:
        ax.scatter(batch[1:, 0], batch[1:, 1], c="magenta", s=100, marker="D", label="Batch alts", **sc_kw)
    ax.set(title="GP predicted mean", xlabel=x_label, ylabel=y_label)
    ax.legend(loc="lower right", fontsize=8, framealpha=0.9)

    ax = axes[1]
    cf2 = ax.contourf(x_mesh, y_mesh, std_mesh, levels=40, cmap="inferno")
    fig.colorbar(cf2, ax=ax, label="Predictive standard deviation", shrink=0.85)
    ax.scatter(X_train[:, 0], X_train[:, 1], c="white", s=45, marker="o", label="Observations", **sc_kw)
    ax.set(title="GP uncertainty", xlabel=x_label, ylabel=y_label)
    ax.legend(loc="lower right", fontsize=8, framealpha=0.9)

    ax = axes[2]
    cf3 = ax.contourf(x_mesh, y_mesh, ei_mesh, levels=40, cmap="plasma")
    fig.colorbar(cf3, ax=ax, label="Expected Improvement", shrink=0.85)
    ax.scatter(X_train[:, 0], X_train[:, 1], c="white", s=45, marker="o", label="Observations", **sc_kw)
    ax.scatter(*next_point, c="cyan", s=220, marker="*", label="Best next", **sc_kw)
    if len(batch) > 1:
        ax.scatter(batch[1:, 0], batch[1:, 1], c="magenta", s=100, marker="D", label="Batch alts", **sc_kw)
    ax.set(title="Expected Improvement", xlabel=x_label, ylabel=y_label)
    ax.legend(loc="lower right", fontsize=8, framealpha=0.9)

    plt.show()
